In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"019cd7cc-aa9d-7004-acde-a1214e1396d6\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Sehetna Forecasting Model\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

In [ ]:
!uv pip install torch accelerate pytorch-forecasting pytorch-lightning --quiet

In [ ]:
# import jinja2
# raw_query = """
#     select *
#     from "25_countries_main.csv" as data_df
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# CONSTANTS
# =============================================================================
FEATURES = [
    'temp_squared', 'pm25_ugm3', 'heat_wave_days', 'month_sin',
    'temp_precip_interaction', 'aqi_pm', 'temp_anomaly_celsius',
    'pollution_vulnerability', 'flood_indicator', 'healthcare_access_index',
    'distance_to_equator', 'pm25_change_rate', 'pm25_ugm3_lag_1w', 'month_cos',
    'temp_change_rate', 'temperature_celsius', 'gdp_per_capita_usd',
    'quarter', 'spatial_lag_pm25', 'is_northern',
    'is_tropical', 'spatial_lag_temp', 'food_security_index',
]

TARGETS = [
    'respiratory_disease_rate', 'cardio_mortality_rate',
    'vector_disease_risk_score', 'waterborne_disease_incidents',
    'heat_related_admissions',
]

SEQ_LEN = 78
HORIZON_LEN = 12
NUM_FEATURES = len(FEATURES)
NUM_TARGETS = len(TARGETS)
ALL_COLS = FEATURES + TARGETS

# Covariate classification for TFT
KNOWN_REALS = [
    'month_sin', 'month_cos', 'quarter',
    'is_northern', 'is_tropical', 'distance_to_equator',
]
UNKNOWN_REALS = [f for f in FEATURES if f not in KNOWN_REALS]

# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
df = data_df.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country_code', 'date']).reset_index(drop=True)

df['temp_squared'] = df['temperature_celsius'] ** 2
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter'] = ((df['month'] - 1) // 3 + 1).astype(float)
df['temp_precip_interaction'] = df['temperature_celsius'] * df['precipitation_mm']
df['pollution_vulnerability'] = df['pm25_ugm3'] / (df['healthcare_access_index'] + 1e-6)
df['distance_to_equator'] = df['latitude'].abs()
df['is_northern'] = (df['latitude'] > 0).astype(float)
df['is_tropical'] = (df['latitude'].abs() < 23.5).astype(float)

df['pm25_change_rate'] = df.groupby('country_code')['pm25_ugm3'].diff()
df['temp_change_rate'] = df.groupby('country_code')['temperature_celsius'].diff()
df['pm25_ugm3_lag_1w'] = df.groupby('country_code')['pm25_ugm3'].shift(1)

for col, new_col in [('pm25_ugm3', 'spatial_lag_pm25'), ('temperature_celsius', 'spatial_lag_temp')]:
    week_sum = df.groupby('date')[col].transform('sum')
    week_count = df.groupby('date')[col].transform('count')
    df[new_col] = (week_sum - df[col]) / (week_count - 1)

for col in ['pm25_change_rate', 'temp_change_rate', 'pm25_ugm3_lag_1w']:
    df[col] = df.groupby('country_code')[col].bfill()

# Integer time index per country (required by pytorch_forecasting)
df['time_idx'] = df.groupby('country_code').cumcount()

# Validate
assert set(FEATURES).issubset(df.columns), f"Missing: {set(FEATURES) - set(df.columns)}"
assert set(TARGETS).issubset(df.columns), f"Missing: {set(TARGETS) - set(df.columns)}"
assert df[ALL_COLS].isna().sum().sum() == 0, "NaN found in features/targets"

df_engineered = df

print(f"Shape: {df_engineered.shape[0]} rows × {df_engineered.shape[1]} cols")
print(f"Features: {NUM_FEATURES} | Targets: {NUM_TARGETS}")
print(f"Countries: {df['country_code'].nunique()} | Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"time_idx range: 0 → {df['time_idx'].max()}")
print(f"Known reals: {len(KNOWN_REALS)} | Unknown reals: {len(UNKNOWN_REALS)}")


Shape: 14100 rows × 47 cols
Features: 23 | Targets: 5
Countries: 25 | Date range: 2015-01-04 → 2025-10-19
time_idx range: 0 → 563
Known reals: 6 | Unknown reals: 17


In [ ]:
# =============================================================================
# TRAIN / VAL / TEST SPLIT — by time_idx cutoffs (80/10/10 per country)
# =============================================================================
max_time_idx = df_engineered['time_idx'].max()  # 563
train_cutoff = int(max_time_idx * 0.80)         # 450
val_cutoff = int(max_time_idx * 0.90)           # 507

train_df = df_engineered[df_engineered['time_idx'] <= train_cutoff].copy()
val_df = df_engineered[(df_engineered['time_idx'] > train_cutoff) &
                        (df_engineered['time_idx'] <= val_cutoff)].copy()
test_df = df_engineered[df_engineered['time_idx'] > val_cutoff].copy()

print(f"Train: {len(train_df):,} rows (time_idx 0–{train_cutoff})")
print(f"Val:   {len(val_df):,} rows  (time_idx {train_cutoff+1}–{val_cutoff})")
print(f"Test:  {len(test_df):,} rows  (time_idx {val_cutoff+1}–{max_time_idx})")
print(f"\ntrain_cutoff={train_cutoff} | val_cutoff={val_cutoff}")


Train: 11,275 rows (time_idx 0–450)
Val:   1,400 rows  (time_idx 451–506)
Test:  1,425 rows  (time_idx 507–563)

train_cutoff=450 | val_cutoff=506


In [ ]:
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, MultiNormalizer

# =============================================================================
# TRAINING DATASET — defines schema, scaling, and windowing for all splits
# =============================================================================
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target=TARGETS,
    group_ids=["country_code"],
    max_encoder_length=SEQ_LEN,
    max_prediction_length=HORIZON_LEN,
    time_varying_known_reals=KNOWN_REALS,
    time_varying_unknown_reals=UNKNOWN_REALS + TARGETS,
    target_normalizer=MultiNormalizer(
        [GroupNormalizer(groups=["country_code"], method="robust") for _ in TARGETS]
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# Val + test inherit schema from training dataset
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, pd.concat([train_df, val_df]), min_prediction_idx=train_cutoff + 1
)
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, pd.concat([train_df, val_df, test_df]), min_prediction_idx=val_cutoff + 1
)

# DataLoaders
BATCH_SIZE = 128
train_loader = training_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_loader = validation_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
test_loader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

print(f"Train samples: {len(training_dataset):,} | Val: {len(validation_dataset):,} | Test: {len(test_dataset):,}")
print(f"Batch size: {BATCH_SIZE} | Train batches/epoch: {len(train_loader)}")


Train samples: 9,050 | Val: 1,125 | Test: 1,150
Batch size: 128 | Train batches/epoch: 70


In [ ]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import MultiLoss, QuantileLoss

tft_model = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=2e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.15,
    hidden_continuous_size=32,
    loss=MultiLoss([QuantileLoss() for _ in TARGETS]),
    optimizer="adamw",
    reduce_on_plateau_patience=5,
)

/home/hexuser/.cache/pypoetry/virtualenvs/python-kernel-OtKFaj5M-py3.11/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/hexuser/.cache/pypoetry/virtualenvs/python-kernel-OtKFaj5M-py3.11/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [ ]:
total_params = sum(p.numel() for p in tft_model.parameters())
trainable_params = sum(p.numel() for p in tft_model.parameters() if p.requires_grad)
print(f"TFT — Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(f"Encoder: {SEQ_LEN} steps | Decoder: {HORIZON_LEN} steps | Targets: {NUM_TARGETS}")
print(f"Hidden: 64 | Heads: 4 | Dropout: 0.15")
print(f"Loss: MultiLoss(QuantileLoss × {NUM_TARGETS})")


TFT — Total params: 562,771 | Trainable: 562,771
Encoder: 78 steps | Decoder: 12 steps | Targets: 5
Hidden: 64 | Heads: 4 | Dropout: 0.15
Loss: MultiLoss(QuantileLoss × 5)


In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor

def train_tft(model, train_loader, val_loader, max_epochs=100, patience=25):
    """Train TFT using Lightning Trainer with early stopping."""
    early_stop = EarlyStopping(monitor="val_loss", patience=patience, mode="min")
    lr_monitor = LearningRateMonitor(logging_interval="epoch")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        gradient_clip_val=1.0,
        callbacks=[early_stop, lr_monitor],
        enable_progress_bar=True,
        log_every_n_steps=10,
    )
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    best_model_path = trainer.checkpoint_callback.best_model_path
    best_model = TemporalFusionTransformer.load_from_checkpoint(best_model_path)
    print(f"\nBest model loaded from: {best_model_path}")
    return best_model, trainer


# =============================================
# >>> UNCOMMENT BELOW TO START TFT TRAINING <<<
# >>> (Long-running: ~10-20 min on GPU)     <<<
# =============================================
best_tft, trainer = train_tft(tft_model, train_loader, val_loader, max_epochs=100, patience=25)


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MultiLoss │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleLi… │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmb… │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDi… │  2.6 K │ train │     0 │
│ 4  │ static_variable_selection          │ Variable… │ 78.3 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ Variable… │  226 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ Variable… │ 49.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedRes… │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedRes… │ 16.8 K │ train │     0 │
│ 9  │ static_context_initia

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_tft(model, test_loader, test_dataset, targets):
    """Evaluate TFT on test set — metrics table + built-in prediction plots."""
    # Raw predictions: list of dicts per target with 'prediction' and quantiles
    raw_preds = model.predict(test_loader, mode="raw", return_x=True)

    # Point predictions (median quantile) for metrics
    preds = model.predict(test_loader, mode="prediction")  # [n_samples, horizon, n_targets]
    actuals = torch.cat([y[0] for x, y in iter(test_loader)])  # [n_samples, horizon, n_targets]

    # Per-target metrics
    results = []
    for i, name in enumerate(targets):
        p = preds[..., i].flatten().numpy()
        a = actuals[..., i].flatten().numpy()
        mae = mean_absolute_error(a, p)
        rmse = np.sqrt(mean_squared_error(a, p))
        r2 = r2_score(a, p)

        threshold = np.percentile(np.abs(a), 90)
        peak_mask = np.abs(a) > threshold
        peak_mae = mean_absolute_error(a[peak_mask], p[peak_mask]) if peak_mask.sum() > 0 else float('nan')

        da = np.diff(a.reshape(-1, HORIZON_LEN), axis=1)
        dp = np.diff(p.reshape(-1, HORIZON_LEN), axis=1)
        dir_acc = (np.sign(da) == np.sign(dp)).mean() * 100

        results.append({
            'Target': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
            'R²': round(r2, 4), 'Peak MAE': round(peak_mae, 4), 'Dir Acc %': round(dir_acc, 2),
        })

    metrics_df = pd.DataFrame(results)
    print("=== Test Set Metrics ===")
    print(metrics_df.to_string(index=False))

    # Built-in prediction plots (3 random samples)
    for idx in np.random.choice(len(test_dataset), size=min(3, len(test_dataset)), replace=False):
        fig = model.plot_prediction(raw_preds.x, raw_preds.output, idx=idx, add_loss_to_title=True)
        plt.show()

    # Variable importance
    interpretation = model.interpret_output(raw_preds.output, reduction="sum")
    figs = model.plot_interpretation(interpretation)
    plt.show()

    return metrics_df


# =============================================
# >>> UNCOMMENT BELOW TO RUN EVALUATION    <<<
# >>> (Requires trained model: best_tft)   <<<
# =============================================
# import torch
# metrics_df = evaluate_tft(best_tft, test_loader, test_dataset, TARGETS)
